In [2]:
# %% [markdown]
# # # ETS-ANN Hybrid Model for Bitcoin; forecast horizon = 10
# # # Python version 3.11+

# %% [markdown]
# ## 1. Import Libraries

# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import time 

# Data and Preprocessing
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split 

# ETS-ANN specific
from pycaret.time_series import TSForecastingExperiment, setup, create_model, compare_models, predict_model, get_config
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# Visualization
import plotly.graph_objects as go
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# %% [markdown]
# ## 2. Configuration

# %%
ticker = "XRP-USD" 
start_date = "2017-11-09" 
end_date = "2025-01-01" 

# Define Train/Test Split Ratio for the *initial* training phase
train_split_ratio = 0.80
# Define Validation Split Ratio *within* the initial training residuals for tuning
validation_split_ratio_for_tuning = 0.20

# --- Walk-Forward Forecast Horizon ---
h = 10
print(f"Setting Walk-Forward Horizon to: h = {h}")

# ANN Parameters
lags_ann = [1, 7, 30]
max_lags = max(lags_ann)

# ANN Hyperparameter Tuning Grid
HP_ANN_NEURONS_OPTIONS = [25, 50, 100]
HP_ANN_EPOCHS_OPTIONS = [30, 50] 
HP_ANN_BATCH_SIZE_OPTIONS = [32, 64]
HP_ANN_LR_TUNE = 0.001

# Final ANN Training Configuration if not tuned
FINAL_ANN_EPOCHS = 50  
FINAL_ANN_BATCH_SIZE = 32 
FINAL_ANN_LR = 0.001

# Retraining configuration
RETRAIN_FREQUENCY = 0
# RETRAIN_EPOCHS_ANN = 5

# %% [markdown]
# ## 3. Data Loading and Preparation

# %%
print(f"--- Loading Data for {ticker} ---")
try:
    df_full = yf.download(tickers=[ticker], start=start_date, end=end_date, progress=False)
    if df_full.empty: raise ValueError(f"No data downloaded for {ticker}.")
    if 'Close' not in df_full.columns: raise ValueError(f"'Close' column not found.")
    df_full = df_full[['Close']].copy()
    df_full = df_full.asfreq('D')
    df_full.ffill(inplace=True); df_full.dropna(inplace=True)
    if df_full.empty: raise ValueError(f"Data became empty.")
    print(f"Loaded {len(df_full)} data points for {ticker} from {df_full.index.min()} to {df_full.index.max()}.")
except Exception as e:
    raise ValueError(f"Failed to load data for {ticker}: {e}")

# %% [markdown]
# ## 4. Data Splitting (Adjusted for h-step Evaluation)

# %%
# Split Data into initial training and test sets
n_total = len(df_full)
n_train = int(train_split_ratio * n_total) 
n_test = n_total - n_train - h + 1 

if n_test <= 0:
     raise ValueError(f"Not enough data for walk-forward with h={h}. Need at least {n_train + h} total points.")

train_data_df = df_full[:n_train]
test_data_full = df_full[n_train:] 

print(f"\nInitial Training Data: {n_train} points ({train_data_df.index.min().strftime('%Y-%m-%d')} to {train_data_df.index.max().strftime('%Y-%m-%d')})")
print(f"Test Data Available: {len(test_data_full)} points")
print(f"Number of walk-forward steps (predictions to generate & evaluate): {n_test}")
if h > 0 and len(test_data_full) >= h:
    print(f"Evaluation Period Start (target t+{h}): {test_data_full.index[h-1].strftime('%Y-%m-%d')}")
    print(f"Evaluation Period End (target t+{h}): {test_data_full.index[-1].strftime('%Y-%m-%d')}")
else:
    print("Evaluation Period cannot be determined due to insufficient test data for horizon.")

# %% [markdown]
# ## 5. Initial ETS Model Selection and Fit

# %%
print("\n--- Initial ETS Model Training ---")
start_time_initial_ets = time.time()
exp_ets_initial = TSForecastingExperiment()
# Setup on initial training data only
setup(data=train_data_df, fh=min(len(test_data_full), 30), 
      session_id=123, verbose=False, numeric_imputation_target="ffill")
print("Comparing ETS models...")
best_ets_model_obj = compare_models(include=['ets', 'exp_smooth'], sort='RMSE', n_select=1, verbose=False)
print(f"Initial ETS Model Selected: {best_ets_model_obj}")
initial_ets_model_fitted = create_model(best_ets_model_obj, verbose=False)
end_time_initial_ets = time.time()
print(f"Initial ETS training finished in {end_time_initial_ets - start_time_initial_ets:.2f} seconds.")

# %% [markdown]
# ## 6. Initial Residual Calculation and Scaling

# %%
print("\n--- Calculating Initial Residuals ---")
y_train_pycaret = get_config('y_train')
try:
    ets_fitted_values_train = initial_ets_model_fitted._fitted_forecaster.fittedvalues
    ets_fitted_values_train = ets_fitted_values_train.reindex(y_train_pycaret.index).dropna()
except AttributeError:
    print("Warning: Fallback - Predicting ETS on initial train data.")
    ets_fitted_values_train = predict_model(initial_ets_model_fitted, data=y_train_pycaret)['y_pred']
    ets_fitted_values_train = ets_fitted_values_train.reindex(y_train_pycaret.index).dropna()

y_train_pycaret_aligned = y_train_pycaret.reindex(ets_fitted_values_train.index)
residuals_initial_train = y_train_pycaret_aligned - ets_fitted_values_train
residuals_initial_train.dropna(inplace=True)
print(f"Calculated {len(residuals_initial_train)} initial residuals.")

print("\n--- Scaling Initial Residuals ---")
scaler_residuals = MinMaxScaler(feature_range=(-1, 1))
normalized_residuals_initial_train = scaler_residuals.fit_transform(residuals_initial_train.values.reshape(-1, 1))
normalized_residuals_initial_train_series = pd.Series(normalized_residuals_initial_train.flatten(), index=residuals_initial_train.index)
print("Residual scaler fitted.")

# %% [markdown]
# ## 7. Prepare Lagged Residual Data for ANN

# %%
print("\n--- Preparing Lagged Residual Features ---")
def create_lagged_features_ann(series, lags):
    lagged_data = pd.DataFrame(index=series.index)
    lagged_data['target'] = series
    for lag in lags:
        lagged_data[f'lag_{lag}'] = series.shift(lag)
    lagged_data.dropna(inplace=True)
    return lagged_data

lagged_norm_resid_initial = create_lagged_features_ann(normalized_residuals_initial_train_series, lags_ann)
X_ann_initial_features = lagged_norm_resid_initial.drop('target', axis=1)
y_ann_initial_target = lagged_norm_resid_initial['target']
print(f"Created lagged residual dataset with shape: {X_ann_initial_features.shape}")

# %% [markdown]
# ## 8. ANN Hyperparameter Tuning (on Initial Residuals)

# %%
print("\n--- Starting ANN Hyperparameter Tuning ---")
start_time_tuning = time.time()
# Split the initial residual data for tuning
X_ann_train_tune, X_ann_val_tune, y_ann_train_tune, y_ann_val_tune = train_test_split(
    X_ann_initial_features, y_ann_initial_target,
    test_size=validation_split_ratio_for_tuning, shuffle=False
)
print(f"ANN Tuning Train shape: {X_ann_train_tune.shape}")
print(f"ANN Tuning Validation shape: {X_ann_val_tune.shape}")
best_val_mse_tune = float('inf')
best_params_ann = None

for neurons in HP_ANN_NEURONS_OPTIONS:
    for epochs in HP_ANN_EPOCHS_OPTIONS:
        for batch_size in HP_ANN_BATCH_SIZE_OPTIONS:
            print(f"Tuning Trial: Neurons={neurons}, Epochs={epochs}, Batch Size={batch_size}")
            ann_model_tune = Sequential([ Dense(neurons, activation='relu', input_shape=(X_ann_train_tune.shape[1],)),
                                          Dense(max(10, neurons//2), activation='relu'), Dense(1) ])
            ann_model_tune.compile(optimizer=Adam(learning_rate=HP_ANN_LR_TUNE), loss='mse')
            history = ann_model_tune.fit(X_ann_train_tune.values, y_ann_train_tune.values, epochs=epochs,
                                         batch_size=batch_size, validation_data=(X_ann_val_tune.values, y_ann_val_tune.values),
                                         verbose=0)
            if 'val_loss' in history.history and len(history.history['val_loss']) > 0:
                 val_mse = history.history['val_loss'][-1]
                 print(f"  Validation MSE: {val_mse:.6f}")
                 if val_mse < best_val_mse_tune:
                     best_val_mse_tune = val_mse
                     best_params_ann = {'neurons': neurons, 'epochs': epochs, 'batch_size': batch_size}
            else: print("  Warning: No validation loss recorded.")

end_time_tuning = time.time()
print(f"\n--- ANN Tuning Complete in {end_time_tuning - start_time_tuning:.2f} seconds ---")
if best_params_ann is None:
     print("Warning: ANN Tuning failed. Using defaults.")
     best_params_ann = {'neurons': HP_ANN_NEURONS, 'epochs': FINAL_ANN_EPOCHS, 'batch_size': FINAL_ANN_BATCH_SIZE}
else:
     print(f"Best Hyperparameters found: {best_params_ann}")
     print(f"Best Validation MSE during tuning: {best_val_mse_tune:.6f}")

# %% [markdown]
# ## 9. Train Final Initial ANN Model

# %%
print("\n--- Training Final Initial ANN Model on Residuals ---")
start_time_initial_ann = time.time()
# Build the final initial ANN model
final_ann_model = Sequential([
    Dense(best_params_ann['neurons'], activation='relu', input_shape=(X_ann_initial_features.shape[1],)),
    Dense(max(10, best_params_ann['neurons']//2), activation='relu'),
    Dense(1)
])
final_ann_model.compile(optimizer=Adam(learning_rate=FINAL_ANN_LR), loss='mse')
print(f"Training final initial ANN...")
# Train on the *entire* initial lagged residual dataset
final_ann_model.fit(X_ann_initial_features.values, y_ann_initial_target.values,
                    epochs=best_params_ann['epochs'], batch_size=best_params_ann['batch_size'], verbose=0)
end_time_initial_ann = time.time()
print(f"Final Initial ANN training complete in {end_time_initial_ann - start_time_initial_ann:.2f} seconds.")
final_ann_model.summary()

# %% [markdown]
# ## 10. Prepare for Walk-Forward Loop

# %%
print("\n--- Preparing for Walk-Forward ---")
try:
    fitted_ets_forecaster = initial_ets_model_fitted._fitted_forecaster
    print(f"Using underlying forecaster: {type(fitted_ets_forecaster)}")
except AttributeError:
    raise AttributeError("Could not access the underlying '_fitted_forecaster' object.")
# Initialize residual history
history_norm_residuals = normalized_residuals_initial_train_series.tolist()
if len(history_norm_residuals) < max_lags:
     print(f"Warning: Padding initial residual history.")
     history_norm_residuals = [0.0] * (max_lags - len(history_norm_residuals)) + history_norm_residuals
print(f"Initial residual history length: {len(history_norm_residuals)}")
ets_ann_walk_forward_predictions_h_step = []

# %% [markdown]
# ## 11. Walk-Forward Validation (Rolling Forecast) Loop - t+h Steps Ahead

# %%
# %% [markdown]
# ## 11. Walk-Forward Validation (Rolling Forecast) Loop - t+h Steps Ahead

# %%
print(f"\n--- Starting ETS-ANN Walk-Forward Validation for {n_test} steps (Predicting {h} steps ahead) ---")
start_time_walk_forward = time.time()

# Use the index for easier tracking, ensure we only iterate n_test times
test_indices_for_loop = test_data_full.index[:n_test]

# Initialize history (as before)
history_norm_residuals = normalized_residuals_initial_train_series.tolist()
if len(history_norm_residuals) < max_lags:
     print(f"Warning: Padding initial residual history.")
     history_norm_residuals = [0.0] * (max_lags - len(history_norm_residuals)) + history_norm_residuals

ets_ann_walk_forward_predictions_h_step = [] # Store final h-step predictions

# --- Loop n_test times using enumerate ---
for i, current_loop_date in enumerate(test_indices_for_loop):

    # Index in the FULL dataset for the *last known actual* data point before this iteration's prediction
    # Corresponds to the date 'current_loop_date'
    current_actual_index = n_train + i

    # --- ETS Prediction (Component 1: Predict t+1 to t+h) ---
    ets_forecast_h_steps = np.full(h, np.nan)
    try:
        # Predict h steps starting *after* current_actual_index
        predict_start_index = current_actual_index + 1
        predict_end_index = current_actual_index + h
        ets_forecast_h_steps = fitted_ets_forecaster.predict(start=predict_start_index, end=predict_end_index)
        # Ensure length is exactly h
        if len(ets_forecast_h_steps) != h:
            print(f"Warning: ETS predict length mismatch step {i+1}. Got {len(ets_forecast_h_steps)}, expected {h}. Padding.")
            current_last_price = df_full['Close'].iloc[current_actual_index]
            padded_preds = np.full(h, current_last_price)
            valid_len = min(h, len(ets_forecast_h_steps)); padded_preds[:valid_len] = ets_forecast_h_steps[:valid_len]
            ets_forecast_h_steps = padded_preds
    except Exception as e:
        print(f"Warning: ETS predict failed step {i+1}. Error: {e}. Using fallback.")
        last_known_price = df_full['Close'].iloc[current_actual_index]
        ets_forecast_h_steps = np.full(h, last_known_price)

    # --- ANN Residual Prediction (Component 2: Predict t+1 to t+h iteratively) ---
    ann_pred_h_steps_denorm = np.zeros(h)
    temp_history_norm_residuals = list(history_norm_residuals)
    can_predict_ann_h = True
    for step_h in range(h):
        if len(temp_history_norm_residuals) >= max_lags:
            current_input_features = [temp_history_norm_residuals[-lag] for lag in lags_ann]
            input_vector = np.array(current_input_features).reshape(1, -1)
            ann_pred_next_norm = final_ann_model.predict(input_vector, verbose=0)[0, 0]
            ann_pred_next_denorm = scaler_residuals.inverse_transform([[ann_pred_next_norm]])[0, 0]
            ann_pred_h_steps_denorm[step_h] = ann_pred_next_denorm
            temp_history_norm_residuals.append(ann_pred_next_norm)
        else:
            print(f"Warning: Not enough history ANN step {step_h+1} at main step {i+1}"); ann_pred_h_steps_denorm[step_h:] = 0; break

    # --- Combine Forecasts & Store Target h-step Prediction ---
    final_pred_t_plus_h = ets_forecast_h_steps[h-1] + ann_pred_h_steps_denorm[h-1]
    ets_ann_walk_forward_predictions_h_step.append(final_pred_t_plus_h) # Append exactly once per main loop iteration

    # --- Update *Main* History with ACTUAL value for step `i` ---
    # Index for the data point that just occurred (corresponding to current_loop_date)
    update_index = n_train + i
    actual_price_t = df_full['Close'].iloc[update_index]

    # Calculate ACTUAL residual for this step 't'
    try:
        # Use the forecast for step 't' (which corresponds to start=update_index)
        ets_forecast_t = fitted_ets_forecaster.predict(start=update_index, end=update_index)[0]
    except Exception as e_ets_t:
        print(f"Warning calculating residual: ETS predict failed ({e_ets_t}). Fallback.")
        if i==0: ets_forecast_t = ets_fitted_values_train.iloc[-1]
        else: ets_forecast_t = ets_forecast_h_steps[0] # Use t+1 ETS forecast from current iter

    actual_residual_t = actual_price_t - ets_forecast_t
    try:
        actual_residual_float = float(actual_residual_t)
        actual_residual_t_norm = scaler_residuals.transform([[actual_residual_float]])[0, 0]
    except Exception as e_transform:
        print(f"ERROR transforming residual update step {i+1}: {e_transform}"); actual_residual_t_norm = 0.0

    history_norm_residuals.append(actual_residual_t_norm) # Update main history

    # Optional: Log progress
    if (i + 1) % 100 == 0:
        print(f"ETS-ANN Walk-Forward (h={h}) Step {i+1}/{n_test} complete.")

    # --- Optional: Periodic Retraining ---
    # ...

end_time_walk_forward = time.time()
total_walk_forward_time = end_time_walk_forward - start_time_walk_forward
print(f"\nETS-ANN Walk-Forward (h={h}) finished in {total_walk_forward_time:.2f} seconds.")

ets_ann_walk_forward_predictions_h_step = np.array(ets_ann_walk_forward_predictions_h_step)
print(f"Length of final predictions: {len(ets_ann_walk_forward_predictions_h_step)}") # Should now be n_test
print(f"Number of walk-forward steps performed: {n_test}") # Check if loop count matches

# %% [markdown]
# ## 12. Evaluate Walk-Forward Performance (t+h)

# %%
# Define evaluation metrics function 
def evaluate_forecast(y_true, y_pred, model_name, horizon):
    """Calculates and prints standard evaluation metrics."""
    y_true_flat = y_true.flatten(); y_pred_flat = y_pred.flatten()
    mae = mean_absolute_error(y_true_flat, y_pred_flat)
    mape = mean_absolute_percentage_error(y_true_flat, y_pred_flat)
    rmse = np.sqrt(mean_squared_error(y_true_flat, y_pred_flat))
    try: r2 = r2_score(y_true_flat, y_pred_flat)
    except ValueError: r2 = np.nan
    print(f"\n--- {model_name} Walk-Forward (t+{horizon}) Evaluation Results ---")
    print(f"RMSE: {rmse:.4f}, MAE: {mae:.4f}, MAPE: {mape:.4%}, R²: {r2:.4f}")
    return {'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2}

# Evaluate against the actual unscaled test data, shifted by h-1 steps
y_test_actual_h_step = test_data_full['Close'].values[h-1:]

# Check lengths before evaluation
if len(y_test_actual_h_step) != len(ets_ann_walk_forward_predictions_h_step):
     raise ValueError(f"Length mismatch after loop: Actual evaluation data ({len(y_test_actual_h_step)}) vs Predictions ({len(ets_ann_walk_forward_predictions_h_step)})")

ets_ann_wf_h_results = evaluate_forecast(y_test_actual_h_step, ets_ann_walk_forward_predictions_h_step, f"ETS-ANN ({ticker})", horizon=h)

# %% [markdown]
# ## 13. Visualize Walk-Forward Results (t+h)

# %%
print("\n--- Plotting Walk-Forward Forecasts ---")
prediction_dates = test_data_full.index[h-1:]
if len(prediction_dates) != len(ets_ann_walk_forward_predictions_h_step):
     min_plot_len = min(len(prediction_dates), len(ets_ann_walk_forward_predictions_h_step))
     prediction_dates = prediction_dates[:min_plot_len]
     plot_predictions = ets_ann_walk_forward_predictions_h_step[:min_plot_len]
     plot_actuals = y_test_actual_h_step[:min_plot_len]
else:
     plot_predictions = ets_ann_walk_forward_predictions_h_step
     plot_actuals = y_test_actual_h_step

results_df_wf = pd.DataFrame({
    'Actual': plot_actuals.flatten(),
    f'ETS-ANN (t+{h})': plot_predictions.flatten()
}, index=prediction_dates)

fig = go.Figure()
fig.add_trace(go.Scatter(x=results_df_wf.index, y=results_df_wf['Actual'], mode='lines', name='Actual Price (Test)', line=dict(color='black')))
fig.add_trace(go.Scatter(x=results_df_wf.index, y=results_df_wf[f'ETS-ANN (t+{h})'], mode='lines', name=f'ETS-ANN Walk-Forward (t+{h})', line=dict(color='blue', dash='dot')))
fig.update_layout(
    title=f'ETS-ANN Walk-Forward (t+{h}) Forecast Comparison for {ticker} (Tuned ANN)',
    xaxis_title="Date (Date being forecast)", yaxis_title="Price (USD)", legend_title="Data/Model", template="plotly_white"
)
fig.show()

# %% [markdown]
# ## 14. Walk-Forward Evaluation Period Summary

# %%
print(f"\n--- Walk-Forward Evaluation Summary ---")
print(f"Initial Training Data End Date: {train_data_df.index.max().strftime('%Y-%m-%d')}")
print(f"Walk-Forward Evaluation Period (Test Set Dates): {test_data_full.index.min().strftime('%Y-%m-%d')} to {test_data_full.index.max().strftime('%Y-%m-%d')}")
print(f"Number of Walk-Forward Steps Performed: {n_test}")
print(f"Forecast Horizon Evaluated at each Step: h = {h}")
# Ensure indices exist before formatting dates
if h > 0 and len(test_data_full) >= h:
    print(f"Evaluation Period (Target Dates): {test_data_full.index[h-1].strftime('%Y-%m-%d')} to {test_data_full.index[-1].strftime('%Y-%m-%d')}")
else:
     print("Evaluation Period cannot be determined due to insufficient test data for horizon.")




Setting Walk-Forward Horizon to: h = 10
--- Loading Data for XRP-USD ---
Loaded 2610 data points for XRP-USD from 2017-11-09 00:00:00 to 2024-12-31 00:00:00.

Initial Training Data: 2088 points (2017-11-09 to 2023-07-28)
Test Data Available: 522 points
Number of walk-forward steps (predictions to generate & evaluate): 513
Evaluation Period Start (target t+10): 2023-08-07
Evaluation Period End (target t+10): 2024-12-31

--- Initial ETS Model Training ---
Comparing ETS models...
Initial ETS Model Selected: ExponentialSmoothing(seasonal='mul', sp=15, trend='add')
Initial ETS training finished in 15.31 seconds.

--- Calculating Initial Residuals ---
Calculated 2058 initial residuals.

--- Scaling Initial Residuals ---
Residual scaler fitted.

--- Preparing Lagged Residual Features ---
Created lagged residual dataset with shape: (2028, 3)

--- Starting ANN Hyperparameter Tuning ---
ANN Tuning Train shape: (1622, 3)
ANN Tuning Validation shape: (406, 3)
Tuning Trial: Neurons=25, Epochs=30, B

Model: "sequential_25"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_75 (Dense)                │ (None, 100)            │           400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_76 (Dense)                │ (None, 50)             │         5,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_77 (Dense)                │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,505 (64.48 KB)

 Trainable params: 5,501 (21.49 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 11,004 (42.99 KB)


--- Preparing for Walk-Forward ---
Using underlying forecaster: <class 'statsmodels.tsa.holtwinters.results.HoltWintersResultsWrapper'>
Initial residual history length: 2058

--- Starting ETS-ANN Walk-Forward Validation for 513 steps (Predicting 10 steps ahead) ---
ETS-ANN Walk-Forward (h=10) Step 100/513 complete.
ETS-ANN Walk-Forward (h=10) Step 200/513 complete.
ETS-ANN Walk-Forward (h=10) Step 300/513 complete.
ETS-ANN Walk-Forward (h=10) Step 400/513 complete.
ETS-ANN Walk-Forward (h=10) Step 500/513 complete.

ETS-ANN Walk-Forward (h=10) finished in 331.40 seconds.
Length of final predictions: 513
Number of walk-forward steps performed: 513

--- ETS-ANN (XRP-USD) Walk-Forward (t+10) Evaluation Results ---
RMSE: 0.4881, MAE: 0.2195, MAPE: 20.7424%, R²: -0.1942

--- Plotting Walk-Forward Forecasts ---



--- Walk-Forward Evaluation Summary ---
Initial Training Data End Date: 2023-07-28
Walk-Forward Evaluation Period (Test Set Dates): 2023-07-29 to 2024-12-31
Number of Walk-Forward Steps Performed: 513
Forecast Horizon Evaluated at each Step: h = 10
Evaluation Period (Target Dates): 2023-08-07 to 2024-12-31
